# 01 — Datasets & ground truth (WS1)

Builds every fixture the rest of the benchmark depends on. **No Milvus needed
for this notebook.** Run top-to-bottom once with `SCALE = "1m"`, then again
with `SCALE = "10m"` (all outputs are cached & resumable; the second run only
does the incremental work).

**Pinned sources** (see `benchlib/config.py`, the single source of truth):

| What | Pin |
|---|---|
| Corpus | `mixedbread-ai/wikipedia-embed-en-2023-11` @ `395eebc` — 41,488,110 English-Wikipedia chunks, 1024-d mxbai-embed-large-v1 embeddings, 721 parquet shards |
| Query/QA source | `google-research-datasets/nq_open` @ `5dd9790` — 87,925 train / 3,610 validation, short gold answers |
| Query embedder | `mixedbread-ai/mxbai-embed-large-v1` @ `b33106f` (Matryoshka-trained; queries get the mxbai retrieval prefix) |
| Seed | 42 |

**What it produces** (all sha256-recorded in `data/MANIFEST.json`):

1. Corpus slice as L2-normalized fp32 memmap — `corpus_{scale}_fp32_norm.npy` — plus an id/title/shard-position parquet. The 1M slice is a **strict prefix** of the 10M slice (deterministic shard order, no shuffling).
2. Query sets: 10,000 GT queries sampled (seed 42) from NQ-Open *train* (dedup'd against validation); NQ-Open *validation* kept untouched for WS4. Both embedded locally.
3. Exact brute-force inner-product **top-100 ground truth** for both query sets at each scale — the recall denominator for every later notebook. Chunked, checkpointed, resumable.
4. A verification section: norm checks, GT self-consistency, retrieval spot-checks, and an **answer-presence@100** measurement (the WS4 feasibility signal, and our measure of NQ↔2023-11-snapshot drift).

**Budget at a glance** (Apple-Silicon Mac, fast connection):

| Step | 1M | 10M | RAM peak |
|---|---|---|---|
| Download shards | ~2.3 GB, 5–15 min | ~23 GB, 45–120 min | <1 GB |
| Build slice memmap | ~4 GB disk, ~5 min | ~41 GB disk, 30–60 min | <2 GB |
| Embed 13.6k queries | 5–15 min on MPS (once, scale-independent) | — | ~3 GB |
| **Ground truth** | **~30–90 min MPS fp32** | **~6–16 h MPS fp32 (overnight) · ~2–4 h with fp16 prefilter · ~30–60 min on an A100/4090** | ~6 GB |
| Verification | ~5 min | ~10 min | ~4 GB |

Total disk for the 10M pass: **~80 GB free** recommended (shards + memmap + GT). Do **not** attempt 10M ground truth on CPU (days).

In [1]:
# ---- 0. Environment & config ------------------------------------------------
import json, math, platform, sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root

import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm

from benchlib import manifest
from benchlib.config import (
    CORPUS_REPO, CORPUS_REVISION, DATA_DIR, EMBED_DIM, EMBED_MODEL,
    EMBED_MODEL_REVISION, GT_CORPUS_CHUNK, GT_PREFILTER_K, GT_QUERY_BLOCK,
    GT_TOP_K, GT_USE_FP16_PREFILTER, N_GT_QUERIES, NQ_REPO, NQ_REVISION,
    QUERY_PREFIX, RESULTS_DIR, SEED, SLICES, gt_path, ids_path, slice_path,
)
from benchlib.download import ensure_rows, fetch_nq

# ======== the one knob: run once with "1m", then again with "10m" ========
SCALE = "10m"
# =========================================================================

N_ROWS = SLICES[SCALE]
DEV = ("cuda" if torch.cuda.is_available()
       else "mps" if torch.backends.mps.is_available() else "cpu")
rng = np.random.default_rng(SEED)

print(f"python {platform.python_version()} on {platform.platform()}")
print(f"torch {torch.__version__}, device = {DEV}")
print(f"RAM: {psutil.virtual_memory().total/2**30:.0f} GB total, "
      f"{psutil.virtual_memory().available/2**30:.0f} GB available")
print(f"free disk at data/: {psutil.disk_usage(str(DATA_DIR.parent)).free/2**30:.0f} GB")
print(f"SCALE = {SCALE} ({N_ROWS:,} vectors, {EMBED_DIM}d)  seed = {SEED}")
if DEV == "cpu":
    print("WARNING: no GPU found. 1M ground truth on CPU takes many hours; "
          "10M is not feasible. Use an Apple-Silicon or CUDA machine.")

python 3.14.4 on macOS-26.5.1-arm64-arm-64bit-Mach-O
torch 2.13.0, device = mps
RAM: 32 GB total, 19 GB available
free disk at data/: 207 GB
SCALE = 10m (10,000,000 vectors, 1024d)  seed = 42


## 1. Download corpus shards *(1M: ~2.3 GB · 10M: ~23 GB)*

Shards are fetched **in order at the pinned revision** until the cumulative
row count covers the slice — the slice is defined as the first `N` rows of
the dataset in shard order. No shuffling: deterministic, and the 1M slice is
a strict prefix of the 10M slice. (Chunk order in this dump carries no
semantic ordering we care about; ANN difficulty is unaffected by row order.)
Each shard's sha256 goes into `data/MANIFEST.json`. Safe to interrupt and
re-run — existing shards are skipped.

In [2]:
# ---- 1. Download shards -----------------------------------------------------
t0 = time.time()
shards = ensure_rows(N_ROWS)
dl_gb = sum(Path(s["path"]).stat().st_size for s in shards) / 2**30
print(f"{len(shards)} shards, {shards[-1]['cum_rows']:,} rows on disk, "
      f"{dl_gb:.1f} GB, {time.time()-t0:.0f}s")

corpus rows:   0%|          | 0/10000000 [00:00<?, ?rows/s]

174 shards, 10,012,482 rows on disk, 21.1 GB, 9s


## 2. Build the normalized fp32 slice memmap *(1M: ~4 GB disk · 10M: ~41 GB)*

Stream the shards, cast embeddings to fp32 (the parquet may store them at
lower/higher precision — we detect and record the source dtype in the
manifest), **L2-normalize** so inner product == cosine everywhere downstream
(Milvus collections will use `metric_type="IP"`), and write one contiguous
`float32` memmap of shape `(N, 1024)`. Row *r* of the memmap ↔ row *r* of the
ids parquet (`_id`, `title`, `shard`, `row_in_shard`) — passage *text* is not
duplicated; it's re-read from the shards on demand.

Streaming, RAM stays <2 GB. Skipped instantly if the output already exists
and matches the manifest.

In [3]:
# ---- 2. Slice memmap + ids parquet -----------------------------------------
SLICE_PATH, IDS_PATH = slice_path(SCALE), ids_path(SCALE)

if SLICE_PATH.exists() and IDS_PATH.exists():
    print("slice already built — skipping (delete files to rebuild)")
else:
    t0 = time.time()
    mm = np.lib.format.open_memmap(SLICE_PATH, mode="w+", dtype=np.float32,
                                   shape=(N_ROWS, EMBED_DIM))
    ids_schema = pa.schema([("_id", pa.string()), ("title", pa.string()),
                            ("shard", pa.int32()), ("row_in_shard", pa.int32())])
    ids_writer = pq.ParquetWriter(IDS_PATH, ids_schema)
    written, src_dtypes = 0, set()
    for s in tqdm(shards, desc="shards"):
        pf = pq.ParquetFile(s["path"])
        row_in_shard = 0
        for batch in pf.iter_batches(batch_size=8192,
                                     columns=["_id", "title", "emb"]):
            col = batch.column("emb")
            src_dtypes.add(str(col.type))
            flat = col.flatten().to_numpy(zero_copy_only=False)
            assert flat.size % EMBED_DIM == 0, "ragged embedding lists?"
            mat = flat.reshape(-1, EMBED_DIM).astype(np.float32, copy=False)
            norms = np.linalg.norm(mat, axis=1, keepdims=True)
            assert norms.min() > 0, "zero-norm embedding found"
            mat = mat / norms
            take = min(len(mat), N_ROWS - written)
            mm[written:written + take] = mat[:take]
            ids_writer.write_batch(pa.record_batch(
                [batch.column("_id").slice(0, take),
                 batch.column("title").slice(0, take),
                 pa.array(np.full(take, s["shard"], dtype=np.int32)),
                 pa.array(np.arange(row_in_shard, row_in_shard + take,
                                    dtype=np.int32))],
                schema=ids_schema))
            written += take
            row_in_shard += len(mat)
            if written >= N_ROWS:
                break
        if written >= N_ROWS:
            break
    mm.flush(); del mm
    ids_writer.close()
    assert written == N_ROWS, f"only {written:,} rows written"
    manifest.record(SLICE_PATH, extra={"rows": N_ROWS, "dim": EMBED_DIM,
                                       "normalized": True,
                                       "source_dtype": sorted(src_dtypes),
                                       "repo": CORPUS_REPO,
                                       "revision": CORPUS_REVISION})
    manifest.record(IDS_PATH)
    print(f"built in {time.time()-t0:.0f}s; source emb dtype(s): {src_dtypes}")
    print("NOTE: if the source dtype is below float32, our fp32 corpus is a "
          "faithful cast of the published vectors — every arm (baseline and "
          "compressed) uses these same vectors, so recall comparisons are "
          "internally consistent.")

slice already built — skipping (delete files to rebuild)


## 3. Query sets *(fast, scale-independent — cached after the 1M run)*

Two disjoint sets from NQ-Open (pinned revision):

- **GT queries** — 10,000 questions sampled with seed 42 from *train*, after dropping duplicates and any question that also appears in *validation*. These drive every recall measurement in WS2/WS3.
- **WS4 queries** — the untouched 3,610-question *validation* split, reserved for the end-task answer-quality eval so it is never used for tuning.

Both are embedded with `mxbai-embed-large-v1` **locally** (queries must use
the mxbai retrieval prefix, documents don't — you cannot reuse corpus vectors
as queries). ~1.3 GB model download on first run; 13,610 texts ≈ 5–15 min on
MPS, ~2 min on CUDA, ~1–2 h on CPU.

In [4]:
# ---- 3a. Sample queries -----------------------------------------------------
Q_GT_PATH = DATA_DIR / "queries_gt.parquet"
Q_WS4_PATH = DATA_DIR / "queries_ws4.parquet"

if Q_GT_PATH.exists() and Q_WS4_PATH.exists():
    q_gt = pd.read_parquet(Q_GT_PATH); q_ws4 = pd.read_parquet(Q_WS4_PATH)
    print(f"cached: {len(q_gt):,} GT queries, {len(q_ws4):,} WS4 queries")
else:
    nq = fetch_nq()
    train = pd.read_parquet(nq["train"])       # columns: question, answer
    val = pd.read_parquet(nq["validation"])
    n0 = len(train)
    train = train.drop_duplicates("question")
    train = train[~train["question"].isin(set(val["question"]))].reset_index(drop=True)
    print(f"train: {n0:,} -> {len(train):,} after dedup/val-overlap removal")
    sel = np.sort(rng.choice(len(train), size=N_GT_QUERIES, replace=False))
    q_gt = train.iloc[sel].reset_index(drop=True)
    q_ws4 = val.reset_index(drop=True)
    q_gt.to_parquet(Q_GT_PATH); q_ws4.to_parquet(Q_WS4_PATH)
    manifest.record(Q_GT_PATH, extra={"seed": SEED, "n": len(q_gt),
                                      "repo": NQ_REPO, "revision": NQ_REVISION})
    manifest.record(Q_WS4_PATH, extra={"split": "validation", "n": len(q_ws4)})
    print(f"sampled {len(q_gt):,} GT queries; reserved {len(q_ws4):,} for WS4")
q_gt.head(3)

cached: 10,000 GT queries, 3,610 WS4 queries


,question,answer
0,where did they film hot tub time machine,[Fernie Alpine Resort]
1,who wrote the music for somewhere in time,[John Barry]
2,when did india participate in olympics for fir...,[1900]


In [5]:
# ---- 3b. Embed queries (mxbai query prefix, normalized fp32) ---------------
QE_GT_PATH = DATA_DIR / "queries_gt_emb.npy"
QE_WS4_PATH = DATA_DIR / "queries_ws4_emb.npy"

if QE_GT_PATH.exists() and QE_WS4_PATH.exists():
    qe_gt = np.load(QE_GT_PATH); qe_ws4 = np.load(QE_WS4_PATH)
    print(f"cached: {qe_gt.shape} / {qe_ws4.shape}")
else:
    from sentence_transformers import SentenceTransformer
    t0 = time.time()
    model = SentenceTransformer(EMBED_MODEL, revision=EMBED_MODEL_REVISION,
                                device=DEV)
    def embed(questions):
        return model.encode([QUERY_PREFIX + q for q in questions],
                            batch_size=64, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=True
                            ).astype(np.float32)
    qe_gt = embed(q_gt["question"].tolist())
    qe_ws4 = embed(q_ws4["question"].tolist())
    np.save(QE_GT_PATH, qe_gt); np.save(QE_WS4_PATH, qe_ws4)
    manifest.record(QE_GT_PATH, extra={"model": EMBED_MODEL,
                                       "revision": EMBED_MODEL_REVISION,
                                       "prefix": QUERY_PREFIX})
    manifest.record(QE_WS4_PATH, extra={"model": EMBED_MODEL,
                                        "revision": EMBED_MODEL_REVISION})
    del model
    print(f"embedded {len(qe_gt)+len(qe_ws4):,} queries in {time.time()-t0:.0f}s")
assert qe_gt.shape == (len(q_gt), EMBED_DIM)
assert np.allclose(np.linalg.norm(qe_gt[:100], axis=1), 1.0, atol=1e-3)

cached: (10000, 1024) / (3610, 1024)


## 4. Exact brute-force top-100 ground truth *(the big one)*

Exact inner-product search of every query against every corpus vector,
streamed in chunks of 200k corpus rows × 2,048 query columns, keeping a
running top-100 per query. **Checkpointed every 10 chunks** — interrupt and
re-run freely; it resumes where it stopped.

Cost is `2·N·Q·D` FLOPs ≈ 2.8×10¹⁶ at 1M / 2.8×10¹⁷ at 10M for our 13,610
queries. Expect (fp32): **1M ≈ 30–90 min on Apple-Silicon MPS**, minutes on a
big CUDA card; **10M ≈ 6–16 h on MPS (run overnight)**, 30–60 min on
A100/4090. Peak RAM ~6 GB (drop `GT_QUERY_BLOCK` to 512 on a 16 GB machine).

`GT_USE_FP16_PREFILTER = True` (in `benchlib/config.py`) roughly halves GPU
time: scores are computed in fp16, the top-500 candidates per query are then
**exactly rescored in fp32**. With a 5× candidate margin over k=100 this is
exact in practice; section 5 verifies it against pure fp32 on a subsample —
if that check ever fails, rerun with the prefilter off.

In [6]:
# ---- 4a. Exact top-k implementation ----------------------------------------
import warnings
warnings.filterwarnings("ignore", message="The given NumPy array is not writable")
# (we only ever *read* the corpus memmap through torch)
def exact_topk(corpus_npy: Path, n_rows: int, queries: np.ndarray, k: int,
               corpus_chunk: int = GT_CORPUS_CHUNK,
               query_block: int = GT_QUERY_BLOCK,
               device: str = DEV,
               fp16_prefilter: bool = GT_USE_FP16_PREFILTER,
               prefilter_k: int = GT_PREFILTER_K,
               ckpt: Path | None = None, ckpt_every: int = 10):
    """Exact (or fp16-prefiltered + fp32-rescored) inner-product top-k of
    `queries` (Q,D fp32, unit-norm) vs the corpus memmap (n_rows,D fp32,
    unit-norm). Returns (indices int64 (Q,k), scores fp32 (Q,k)), sorted
    descending. Resumable via `ckpt`."""
    corpus = np.lib.format.open_memmap(corpus_npy, mode="r")
    assert corpus.shape[0] >= n_rows and corpus.shape[1] == queries.shape[1]
    Q, D = queries.shape
    qt = torch.from_numpy(np.ascontiguousarray(queries))
    n_chunks = math.ceil(n_rows / corpus_chunk)
    top_s = torch.full((Q, k), -2.0)                 # cosine ∈ [-1,1]
    top_i = torch.full((Q, k), -1, dtype=torch.int64)
    start = 0
    if ckpt and ckpt.exists():
        ck = np.load(ckpt)
        start = int(ck["next_chunk"])
        top_s = torch.from_numpy(ck["scores"]); top_i = torch.from_numpy(ck["idx"])
        print(f"  resuming at chunk {start}/{n_chunks}")
    qdev = qt.to(device)
    for ci in tqdm(range(start, n_chunks), desc="corpus chunks",
                   initial=start, total=n_chunks):
        lo, hi = ci * corpus_chunk, min((ci + 1) * corpus_chunk, n_rows)
        C = hi - lo
        cb = torch.from_numpy(np.ascontiguousarray(corpus[lo:hi])).to(device)
        cb16 = cb.half() if fp16_prefilter else None
        for qlo in range(0, Q, query_block):
            qhi = min(qlo + query_block, Q)
            qs = qdev[qlo:qhi]                                    # (q, D) fp32
            if fp16_prefilter:
                s16 = (qs.half() @ cb16.T).float()                # (q, C)
                kk = min(prefilter_k, C)
                _, cand = torch.topk(s16, kk, dim=1)              # fp16 ranks
                exact = torch.empty(qhi - qlo, kk, device=device)
                for slo in range(0, qhi - qlo, 256):              # fp32 rescore
                    shi = min(slo + 256, qhi - qlo)
                    g = cb[cand[slo:shi].reshape(-1)].reshape(shi - slo, kk, D)
                    exact[slo:shi] = torch.einsum("qd,qkd->qk", qs[slo:shi], g)
                keep = min(k, kk)
                cs, ci_local = torch.topk(exact, keep, dim=1)
                cidx = torch.gather(cand, 1, ci_local) + lo
            else:
                s = qs @ cb.T                                     # (q, C) fp32
                keep = min(k, C)
                cs, ci_ = torch.topk(s, keep, dim=1)
                cidx = ci_ + lo
            ms = torch.cat([top_s[qlo:qhi].to(device), cs], dim=1)
            mi = torch.cat([top_i[qlo:qhi].to(device), cidx], dim=1)
            bs, border = torch.topk(ms, k, dim=1)
            top_s[qlo:qhi] = bs.cpu()
            top_i[qlo:qhi] = torch.gather(mi, 1, border).cpu()
        del cb, cb16
        if ckpt and ((ci + 1) % ckpt_every == 0 or ci + 1 == n_chunks):
            np.savez(ckpt, next_chunk=ci + 1, scores=top_s.numpy(),
                     idx=top_i.numpy())
    assert (top_i >= 0).all(), "unfilled ground-truth slots"
    return top_i.numpy(), top_s.numpy().astype(np.float32)


In [7]:
# ---- 4b. Run ground truth for both query sets at this SCALE ----------------
GT_PATH = gt_path(SCALE)

if GT_PATH.exists():
    gt = np.load(GT_PATH)
    print(f"cached ground truth: {GT_PATH.name}",
          {k: v.shape for k, v in gt.items()})
else:
    t0 = time.time()
    all_q = np.concatenate([qe_gt, qe_ws4])          # one pass, both sets
    idx, sc = exact_topk(SLICE_PATH, N_ROWS, all_q, GT_TOP_K,
                         ckpt=DATA_DIR / f"gt_ckpt_{SCALE}.npz")
    n1 = len(qe_gt)
    np.savez(GT_PATH,
             gt_indices=idx[:n1], gt_scores=sc[:n1],
             ws4_indices=idx[n1:], ws4_scores=sc[n1:])
    (DATA_DIR / f"gt_ckpt_{SCALE}.npz").unlink(missing_ok=True)
    manifest.record(GT_PATH, extra={
        "scale": SCALE, "k": GT_TOP_K, "n_gt_queries": n1,
        "n_ws4_queries": len(qe_ws4), "fp16_prefilter": GT_USE_FP16_PREFILTER,
        "device": DEV, "seconds": round(time.time() - t0)})
    gt = np.load(GT_PATH)
    print(f"done in {(time.time()-t0)/60:.1f} min -> {GT_PATH.name}")

cached ground truth: gt_10m_top100.npz {'gt_indices': (10000, 100), 'gt_scores': (10000, 100), 'ws4_indices': (3610, 100), 'ws4_scores': (3610, 100)}


## 5. Verification *(never skip this)*

Checks, in order: corpus norms; GT structural sanity (sorted scores, no
duplicate indices); **fp16-prefilter equivalence** against pure fp32 on 20
queries (only meaningful if the prefilter was on); retrieval spot-checks you
can eyeball; and **answer-presence@k** — the fraction of GT queries whose
gold answer string appears verbatim (normalized) in the top-k retrieved
passages. Answer-presence is our WS4 feasibility signal *and* the honest
measure of NQ (2018-era questions) vs 2023-11-snapshot drift — it goes into
`results/ws1/` and gets quoted in the talk, not hidden.

In [8]:
# ---- 5a. Structural checks --------------------------------------------------
corpus = np.lib.format.open_memmap(SLICE_PATH, mode="r")
sample = rng.choice(N_ROWS, 1000, replace=False)
norms = np.linalg.norm(corpus[np.sort(sample)], axis=1)
assert np.allclose(norms, 1.0, atol=1e-3), "corpus rows not unit-norm"

gt_i, gt_s = gt["gt_indices"], gt["gt_scores"]
assert gt_i.shape == (len(q_gt), GT_TOP_K)
assert (np.diff(gt_s, axis=1) <= 1e-6).all(), "GT scores not sorted desc"
assert all(len(np.unique(r)) == GT_TOP_K for r in gt_i[:200]), "dup indices"
print("structural checks OK "
      f"(top-1 score mean {gt_s[:,0].mean():.3f}, "
      f"top-100 score mean {gt_s[:,-1].mean():.3f})")

structural checks OK (top-1 score mean 0.770, top-100 score mean 0.660)


In [9]:
# ---- 5b. fp16-prefilter equivalence check (20 queries, pure fp32 re-run) ---
if GT_USE_FP16_PREFILTER:
    probe = rng.choice(len(q_gt), 20, replace=False)
    ref_i, _ = exact_topk(SLICE_PATH, N_ROWS, qe_gt[probe], GT_TOP_K,
                          fp16_prefilter=False, ckpt=None)
    overlap = np.mean([len(set(ref_i[j]) & set(gt_i[probe[j]])) / GT_TOP_K
                       for j in range(len(probe))])
    print(f"fp16-prefilter vs pure-fp32 top-{GT_TOP_K} overlap: {overlap:.4f}")
    assert overlap >= 0.999, ("PREFILTER NOT EXACT ON THIS DATA — "
                              "set GT_USE_FP16_PREFILTER=False and re-run GT")
else:
    print("pure fp32 GT — nothing to verify here")

pure fp32 GT — nothing to verify here


In [10]:
# ---- 5c. Spot-check: eyeball 3 retrievals ----------------------------------
from benchlib.download import shard_local_path

ids = pd.read_parquet(IDS_PATH)

def load_texts(global_rows):
    """Fetch passage text for global row indices (groups reads by shard)."""
    rows = ids.iloc[global_rows][["shard", "row_in_shard"]]
    out = pd.Series(index=rows.index, dtype=object)
    for shard, grp in rows.groupby("shard"):
        tbl = pq.read_table(shard_local_path(int(shard)), columns=["text"])
        texts = tbl.column("text").to_pylist()
        out.loc[grp.index] = [texts[r] for r in grp["row_in_shard"]]
    return out.tolist()

for j in rng.choice(len(q_gt), 3, replace=False):
    print("Q:", q_gt['question'].iloc[j], "| gold:", list(q_gt['answer'].iloc[j]))
    top = gt_i[j, :3]
    for rank, (title, text) in enumerate(zip(
            ids['title'].iloc[top], load_texts(top))):
        print(f"  #{rank+1} [{title}] {text[:140]}...")
    print()

Q: who played the muses in disney's hercules | gold: ['LaChanze', 'Cheryl Freeman', 'Lillias White', 'Vanéese Y. Thomas', 'Roz Ryan']
  #1 [Muses in popular culture] Melpomene features in the 1997 Walt Disney Pictures film Hercules, appearing alongside the muses Calliope, Clio, Terpsichore and Thalia, who...
  #2 [Muses in popular culture] Thalia features in the 1997 Walt Disney Pictures film Hercules, appearing alongside the muses Calliope, Clio, Melpomene and Terpsichore, who...
  #3 [Muses in popular culture] Clio features in the 1997 Walt Disney Pictures film Hercules, appearing alongside the muses Calliope, Melpomene, Terpsichore and Thalia, who...

Q: who made the bomb that was dropped on hiroshima | gold: ['Los Alamos Laboratory']


  #1 [Nuclear technology] Ultimately, the Manhattan Project manufactured nuclear weapons based on each of these elements. They detonated the first nuclear weapon in a...
  #2 [Hiroshima] On Monday, August 6, 1945, at 8:15 a.m. (Hiroshima time), the American Boeing B-29 Superfortress, the Enola Gay, flown by Paul Tibbets (23 F...
  #3 [August 1945] Atomic bombing of Hiroshima: United States B-29 Superfortress Enola Gay dropped a uranium-235 atomic bomb codenamed "Little Boy" on the Japa...

Q: what episode of the west wing does cj sing the jackal | gold: ['Six Meetings Before Lunch']
  #1 [The West Wing (season 6)] The sixth season opens with the Israeli and Palestinian delegations arriving at Camp David for peace talks. Despite problems at the summit, ...
  #2 [West Wing Week] The show is shot with a Sony EX3, a 1980s-era 416 Sennheiser shotgun microphone and subsequently edited on Final Cut Pro. Video footage is r...
  #3 [A Proportional Response] "A Proportional Response" is the thir

In [11]:
# ---- 5d. Answer-presence@k (WS4 feasibility + snapshot-drift measure) ------
import re, unicodedata

def norm_text(s):
    s = unicodedata.normalize("NFKD", s.lower())
    return re.sub(r"[^a-z0-9 ]", " ", s)

N_AP_SAMPLE = 1000        # queries sampled for this check (text I/O bound)
ap_sample = np.sort(rng.choice(len(q_gt), N_AP_SAMPLE, replace=False))
hits = {10: 0, 100: 0}
t0 = time.time()
for j in tqdm(ap_sample, desc="answer presence"):
    answers = [norm_text(a) for a in q_gt["answer"].iloc[j]]
    texts = [norm_text(t) for t in load_texts(gt_i[j])]
    for k in hits:
        blob = " ".join(texts[:k])
        if any(a in blob for a in answers):
            hits[k] += 1
ap = {f"answer_presence@{k}": v / N_AP_SAMPLE for k, v in hits.items()}
print(ap, f"({time.time()-t0:.0f}s)")

out = RESULTS_DIR / "ws1"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"scale": SCALE, "n_sample": N_AP_SAMPLE, "seed": SEED, **ap}]
             ).to_csv(out / f"answer_presence_{SCALE}.csv", index=False)
print(f"-> results/ws1/answer_presence_{SCALE}.csv")
# Interpretation: this is the ceiling for extractive answer quality in WS4.
# If answer_presence@100 is low (<~0.7 at 10M), WS4 needs the drift filter:
# restrict its question set to queries whose answer is present here.

answer presence:   0%|          | 0/1000 [00:00<?, ?it/s]

{'answer_presence@10': 0.558, 'answer_presence@100': 0.677} (510s)
-> results/ws1/answer_presence_10m.csv


## 6. Summary

Everything below is recorded in `data/MANIFEST.json`; re-verify any time with
`python scripts/fixtures/verify_checksums.py`. Re-run this notebook with the other
`SCALE` value if you haven't yet. Then: `02_quantization_curves.ipynb`
(Milvus must be up: `make up`).

In [12]:
# ---- 6. Manifest summary ----------------------------------------------------
m = manifest.load_manifest()
manifest.record_meta(f"nb01_completed_{SCALE}",
                     time.strftime("%Y-%m-%dT%H:%M:%S"))
print(f"{len(m['files'])} files in manifest")
summary = pd.DataFrame([
    {"file": k, "GB": v["bytes"] / 2**30, "sha256": v["sha256"][:16] + "…"}
    for k, v in sorted(m["files"].items()) if "corpus_shards" not in k])
summary.round(2)

186 files in manifest


,file,GB,sha256
0,corpus_10m_fp32_norm.npy,38.15,57533c40b714fcc8…
1,corpus_10m_ids.parquet,0.14,69259e9e78309a64…
2,corpus_1m_fp32_norm.npy,3.81,aa555cef9aae6ab4…
3,corpus_1m_ids.parquet,0.01,60b28142dcd23ce8…
4,gt_10m_top100.npz,0.02,f339f88f6caf67df…
5,gt_1m_top100.npz,0.02,8ea46df945882732…
6,nq/train-00000-of-00001.parquet,0.00,25d3a544324f900b…
7,nq/validation-00000-of-00001.parquet,0.00,b074bed0bccb56fa…
8,queries_gt.parquet,0.00,a8cf1176be855178…
9,queries_gt_emb.npy,0.04,b7237f31f88fc9f5…
